In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- tesseract_concat_lazy ---
FIX_TESSERACT_CONCAT_LAZY_LIST_DFS = [pl.DataFrame({"value":["Hello","World"],"page":[1,1],"x1":[10,110],"y1":[10,10],"x2":[100,200],"y2":[30,30],"conf":[99.5,98.2]}), pl.DataFrame({"value":["Foo"],"page":[2],"x1":[5],"y1":[50],"x2":[50],"y2":[65],"conf":[97.0]})]

# --- tesseract_from_dicts ---
FIX_TESSERACT_FROM_DICTS_LIST_DFS = [pl.DataFrame({"value":["Hello","World"],"page":[1,1],"x1":[10,110],"y1":[10,10],"x2":[100,200],"y2":[30,30],"conf":[99.5,98.2]}), pl.DataFrame({"value":["Foo"],"page":[2],"x1":[5],"y1":[50],"x2":[50],"y2":[65],"conf":[97.0]})]
FIX_TESSERACT_FROM_DICTS_LIST_ELEMENTS = [{"value":"Hello","page":1,"x1":10,"y1":10,"x2":100,"y2":30,"conf":99.5},{"value":"World","page":1,"x1":110,"y1":10,"x2":200,"y2":30,"conf":98.2}]

# --- tesseract_nan_none ---
FIX_TESSERACT_NAN_NONE_D_EL = {"value":"Hello","page":1,"x1":10,"y1":10,"x2":100,"y2":50}

print("✅ Fixtures loaded")
OCRDataframe = SimpleNamespace  # mock for testing


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_tesseract_concat_lazy(list_dfs):
    return OCRDataframe(df=pd.concat(list_dfs))
    return None

def before_tesseract_from_dicts(list_dfs, list_elements):
    list_dfs.append(pd.DataFrame(list_elements))
    return None

def before_tesseract_nan_none(d_el):
    d_el["confidence"] = np.nan
    return None

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_tesseract_concat_lazy(list_dfs):

    return OCRDataframe(df=pl.concat(list_dfs, how="diagonal"))

def gen_tesseract_from_dicts(list_dfs, list_elements):

    list_dfs.append(pl.DataFrame(list_elements))
    return None

def gen_tesseract_nan_none(d_el):

    d_el = d_el.with_columns(pl.lit(float("nan")).alias("confidence"))
    return None

def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)

# ── Test harness type adapters ─────────────────────────────────────────────
def _to_pandas_fixture(obj):
    if isinstance(obj, pl.Series):
        return obj.to_pandas()
    if isinstance(obj, pl.DataFrame):
        return obj.to_pandas()
    if isinstance(obj, pl.LazyFrame):
        return obj.collect().to_pandas()
    if isinstance(obj, list):
        return [_to_pandas_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_pandas_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_pandas_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), (pl.DataFrame, pl.LazyFrame)):
        return SimpleNamespace(df=_to_pandas_fixture(obj.df))
    return obj

def _to_polars_fixture(obj):
    if isinstance(obj, pd.Series):
        return pl.Series(obj.name or "series", obj.to_list())
    if isinstance(obj, pd.DataFrame):
        return pl.from_pandas(obj)
    if isinstance(obj, list):
        return [_to_polars_fixture(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(_to_polars_fixture(x) for x in obj)
    if isinstance(obj, dict):
        return {k: _to_polars_fixture(v) for k, v in obj.items()}
    if hasattr(obj, "df") and isinstance(getattr(obj, "df"), pd.DataFrame):
        return SimpleNamespace(df=_to_polars_fixture(obj.df))
    return obj

def _wrap_before_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_pandas_fixture(self.df)
        try:
            return fn(*[_to_pandas_fixture(a) for a in args], **{k: _to_pandas_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

def _wrap_gen_func(fn):
    def _wrapped(*args, **kwargs):
        _old_self_df = None
        if "self" in globals() and hasattr(self, "df"):
            _old_self_df = self.df
            self.df = _to_polars_fixture(self.df)
        try:
            return fn(*[_to_polars_fixture(a) for a in args], **{k: _to_polars_fixture(v) for k, v in kwargs.items()})
        finally:
            if _old_self_df is not None:
                self.df = _old_self_df
    return _wrapped

for _name, _fn in list(globals().items()):
    if callable(_fn) and _name.startswith("before_"):
        globals()[_name] = _wrap_before_func(_fn)
    elif callable(_fn) and _name.startswith("gen_"):
        globals()[_name] = _wrap_gen_func(_fn)


In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if hasattr(r, "df"):
        r = r.df
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: tesseract_nan_none ===

# L1 smoke – generated
try:
    _r = gen_tesseract_nan_none(FIX_TESSERACT_NAN_NONE_D_EL)
    print("✅ L1 smoke gen_tesseract_nan_none: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_tesseract_nan_none: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_tesseract_nan_none(FIX_TESSERACT_NAN_NONE_D_EL)
    print("✅ L1 smoke before_tesseract_nan_none: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_tesseract_nan_none: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence - compare the dictionary mutation.
try:
    def _raw_func(_fn):
        _seen = set()
        while callable(_fn) and getattr(_fn, "__name__", "") == "_wrapped":
            if id(_fn) in _seen:
                break
            _seen.add(id(_fn))
            _nested = [
                _cell.cell_contents
                for _cell in (getattr(_fn, "__closure__", None) or [])
                if callable(_cell.cell_contents)
            ]
            if not _nested:
                break
            _fn = _nested[0]
        return _fn

    def _same_missing_aware(_a, _b):
        if _a.keys() != _b.keys():
            return False
        return all(
            (pd.isna(_a[_k]) and pd.isna(_b[_k])) or _a[_k] == _b[_k]
            for _k in _a
        )

    _before_d = dict(FIX_TESSERACT_NAN_NONE_D_EL)
    _gen_d = dict(FIX_TESSERACT_NAN_NONE_D_EL)
    _raw_func(before_tesseract_nan_none)(_before_d)
    _raw_func(gen_tesseract_nan_none)(_gen_d)
    if _same_missing_aware(_before_d, _gen_d):
        print("✅ L2 equivalence tesseract_nan_none: MATCH")
    else:
        print(f"❌ L2 equivalence tesseract_nan_none: MISMATCH before={_before_d!r} generated={_gen_d!r}")
except Exception as _e:
    print(f"❌ L2 equivalence tesseract_nan_none: setup error - {type(_e).__name__}: {_e}")

# L3 edge - an empty dictionary still receives a missing confidence value.
try:
    _before_d, _gen_d = {}, {}
    _raw_func(before_tesseract_nan_none)(_before_d)
    _raw_func(gen_tesseract_nan_none)(_gen_d)
    if _same_missing_aware(_before_d, _gen_d):
        print("✅ L3 edge tesseract_nan_none empty dict: MATCH")
    else:
        print(f"❌ L3 edge tesseract_nan_none empty dict: MISMATCH before={_before_d!r} generated={_gen_d!r}")
except Exception as _e:
    print(f"❌ L3 edge tesseract_nan_none empty dict: {type(_e).__name__}: {_e}")


❌ L1 smoke gen_tesseract_nan_none: AttributeError: 'dict' object has no attribute 'with_columns'
✅ L1 smoke before_tesseract_nan_none: OK
❌ L2 equivalence tesseract_nan_none: setup error - AttributeError: 'dict' object has no attribute 'with_columns'
❌ L3 edge tesseract_nan_none empty dict: AttributeError: 'dict' object has no attribute 'with_columns'
